In [1]:
import torch
print(torch.cuda.get_device_name(0))
print(torch.cuda.is_available())

NVIDIA GeForce RTX 5080
True


In [ ]:
import os
import re
import pandas as pd
import numpy as np
import torch
import torchaudio
import evaluate
from datasets import Dataset, Features, Value
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor, TrainingArguments, Trainer

# Local Common Voice (MN) directory (TSVs + clips/)
DATA_DIR = "/home/toru2/Amara/Deep_learning/lab3/common_voice_mn"
CLIPS_DIR = os.path.join(DATA_DIR, "clips")

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

# ---------- Load local TSVs ----------
def load_tsv(split_name: str) -> Dataset:
    tsv_path = os.path.join(DATA_DIR, f"{split_name}.tsv")
    df = pd.read_csv(tsv_path, sep="\t", usecols=["path", "sentence"])
    df = df.dropna(subset=["path", "sentence"]).copy()
    df["path"] = df["path"].astype(str).str.strip().map(lambda p: os.path.join(CLIPS_DIR, p))
    df["sentence"] = df["sentence"].astype(str)
    
    # Force Arrow schema to plain strings (avoids large_string/dictionary issues)
    features = Features({"path": Value("string"), "sentence": Value("string")})
    return Dataset.from_pandas(df, features=features, preserve_index=False)

ds_train = load_tsv("train")
ds_test = load_tsv("test")

print(ds_train)
print(ds_test)

Casting the dataset:   0%|          | 0/2193 [00:00<?, ? examples/s]

In [ ]:
# ---------- Sanity check 1: files exist ----------
def path_exists(p: str) -> bool:
    try:
        return os.path.isfile(p)
    except Exception:
        return False

missing_train = [p for p in ds_train["path"][:200] if not path_exists(p)]
missing_test = [p for p in ds_test["path"][:200] if not path_exists(p)]
print("missing train (first 200):", len(missing_train))
print("missing test  (first 200):", len(missing_test))
if missing_train[:3]:
    print("example missing train paths:", missing_train[:3])
if missing_test[:3]:
    print("example missing test paths:", missing_test[:3])

# Optional: filter out missing files globally
ds_train = ds_train.filter(lambda x: path_exists(x["path"]))
ds_test = ds_test.filter(lambda x: path_exists(x["path"]))
print("after filtering:", ds_train, ds_test)

In [ ]:
# ---------- Sanity check 2: load one audio file ----------
p0 = ds_train[0]["path"]
waveform, sr = torchaudio.load(p0)
print("path:", p0)
print("waveform shape:", waveform.shape, "sr:", sr)

# Convert to mono float32
if waveform.shape[0] > 1:
    waveform = waveform.mean(dim=0, keepdim=True)
waveform = waveform.squeeze(0)
print("mono shape:", waveform.shape)

In [ ]:
# ---------- Text normalization + label prep ----------
chars_to_ignore_regex = r"[\,\?\.\!\-\;\:\"\“\”\’\‘]"

def normalize_text(s: str) -> str:
    s = "" if s is None else str(s)
    s = re.sub(chars_to_ignore_regex, "", s).lower().strip()
    s = re.sub(r"\s+", " ", s)
    return s

print("before:", ds_train[0]["sentence"])
print("after :", normalize_text(ds_train[0]["sentence"]))

# Filter out empty normalized transcripts
ds_train = ds_train.filter(lambda x: len(normalize_text(x["sentence"])) > 0)
ds_test = ds_test.filter(lambda x: len(normalize_text(x["sentence"])) > 0)
print("after text filtering:", ds_train, ds_test)

In [ ]:
# ---------- Load processor/model ----------
processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-large-xlsr-53-mongolian")
model = Wav2Vec2ForCTC.from_pretrained("facebook/wav2vec2-large-xlsr-53-mongolian").to(device)
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.ctc_zero_infinity = True
print("pad_token_id:", processor.tokenizer.pad_token_id)

In [ ]:
# ---------- Audio -> input_values + labels (start with a small subset) ----------
def speech_file_to_array(path: str, target_sr: int = 16000) -> np.ndarray:
    waveform, sr = torchaudio.load(path)
    # waveform: (channels, time)
    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)
    waveform = waveform.squeeze(0)
    if sr != target_sr:
        waveform = torchaudio.functional.resample(waveform, sr, target_sr)
    return waveform.numpy().astype(np.float32)

def prepare_batch(batch):
    text = normalize_text(batch["sentence"])
    speech = speech_file_to_array(batch["path"], target_sr=16000)
    inputs = processor(speech, sampling_rate=16000)
    batch["input_values"] = inputs.input_values[0]
    with processor.as_target_processor():
        batch["labels"] = processor(text).input_ids
    return batch

mini_train = ds_train.select(range(min(50, len(ds_train)))).map(prepare_batch, remove_columns=ds_train.column_names)
print(mini_train)
print("keys:", mini_train[0].keys())
print("len(input_values):", len(mini_train[0]["input_values"]))
print("len(labels):", len(mini_train[0]["labels"]))

In [ ]:
# ---------- Data collator sanity check ----------
from dataclasses import dataclass
from typing import Dict, List, Union

@dataclass
class DataCollatorCTCWithPadding:
    processor: Wav2Vec2Processor
    padding: Union[bool, str] = True

    def __call__(self, features: List[Dict[str, Union[List[int], np.ndarray]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_values": f["input_values"]} for f in features]
        label_features = [{"input_ids": f["labels"]} for f in features]

        batch = self.processor.pad(input_features, padding=self.padding, return_tensors="pt")
        with self.processor.as_target_processor():
            labels_batch = self.processor.pad(label_features, padding=self.padding, return_tensors="pt")

        labels = labels_batch["input_ids"].masked_fill(labels_batch["attention_mask"].ne(1), -100)
        batch["labels"] = labels
        return batch

collator = DataCollatorCTCWithPadding(processor)
b = collator([mini_train[i] for i in range(min(4, len(mini_train)))])
print({k: v.shape for k, v in b.items()})
print("labels contain -100?", bool((b["labels"] == -100).any().item()))

In [ ]:
# ---------- One forward pass sanity check (loss finite) ----------
model.train()
out = model(
    input_values=b["input_values"].to(device),
    attention_mask=b["attention_mask"].to(device),
    labels=b["labels"].to(device),
 )
print("loss:", float(out.loss))
assert torch.isfinite(out.loss).item()

In [ ]:
# ---------- Baseline WER on a small subset (before training) ----------
wer_metric = evaluate.load("wer")
model.eval()

def decode_labels(label_ids: np.ndarray) -> list[str]:
    label_ids = label_ids.copy()
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    return processor.batch_decode(label_ids, group_tokens=False)

def predict_wer(ds: Dataset, n: int = 50, batch_size: int = 8) -> float:
    n = min(n, len(ds))
    ds_small = ds.select(range(n)).map(prepare_batch, remove_columns=ds.column_names)
    preds, refs = [], []
    for start in range(0, n, batch_size):
        feats = [ds_small[i] for i in range(start, min(start + batch_size, n))]
        bb = collator(feats)
        with torch.no_grad():
            logits = model(
                input_values=bb["input_values"].to(device),
                attention_mask=bb["attention_mask"].to(device),
            ).logits
        pred_ids = torch.argmax(logits, dim=-1).cpu().numpy()
        pred_str = processor.batch_decode(pred_ids)
        ref_str = decode_labels(bb["labels"].cpu().numpy())
        preds.extend([normalize_text(s) for s in pred_str])
        refs.extend([normalize_text(s) for s in ref_str])
    return wer_metric.compute(predictions=preds, references=refs)

wer50 = predict_wer(ds_test, n=50, batch_size=8)
print("WER on 50 samples (pre-train):", 100 * wer50)

In [ ]:
# ---------- Full preprocessing + Trainer ----------
ds_train_p = ds_train.map(prepare_batch, remove_columns=ds_train.column_names)
ds_test_p = ds_test.map(prepare_batch, remove_columns=ds_test.column_names)

def compute_metrics(pred):
    pred_ids = np.argmax(pred.predictions, axis=-1)
    pred_str = processor.batch_decode(pred_ids)

    label_ids = pred.label_ids
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    label_str = processor.batch_decode(label_ids, group_tokens=False)

    pred_str = [normalize_text(s) for s in pred_str]
    label_str = [normalize_text(s) for s in label_str]
    return {"wer": wer_metric.compute(predictions=pred_str, references=label_str)}

training_args = TrainingArguments(
    output_dir="./wav2vec2-mn-ft",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    evaluation_strategy="steps",
    save_steps=500,
    eval_steps=500,
    logging_steps=50,
    learning_rate=2e-5,
    num_train_epochs=3,
    fp16=torch.cuda.is_available(),
    dataloader_num_workers=2,
    report_to="none",
 )

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=ds_train_p,
    eval_dataset=ds_test_p,
    data_collator=collator,
    tokenizer=processor.feature_extractor,
    compute_metrics=compute_metrics,
 )

trainer.train()